# **Task 2 — DeiT**: does seeing more pixels help?

This week's seminar was about [DeiT](https://proceedings.mlr.press/v139/touvron21a.html): a ViT trained on ImageNet alone, with distillation, and fine-tuned at a higher resolution at the end. The paper reports that going from 224×224 to 384×384 images makes the model more accurate (Table 10). Today you check whether that holds for **our** model and **our** data.

This notebook loads the data and the model and gives you a **fixed evaluation**, so that every group measures its result the same way. Your own work starts after the baseline.

<details>
<summary>Reach out here if you have any problems</summary>
Taras Kutsyk — taras.kutsyk@doctoral.uj.edu.pl
</details>

# The task

**Question.** Does seeing more pixels help? Make the model run at 384×384, fine-tune it within the budget, and **answer the question for this model and this data, with evidence**. Then explain *why* you got the result you got, and what they teach you.

*Hint:* be wary of what you compare with, and of how your claim relates to the paper's.

**Fixed:**
- the starting model and the dataset;
- the `evaluate()` function below;
- the fine-tuning budget: **at most 100,000 training images per fine-tuned model** (about 11 passes over the training set);
- write the resizing of the position embeddings yourselves: no ready-made resizing (e.g. timm's `img_size=` argument or `resample_abs_pos_embed`).

Everything else is your choice.

**Bring to the lab:** your code + a short deck with your answer, the evidence for it, and your explanation. If your group defends this week, **every member should be able to explain any piece of the code**.

# Setup

**Step 0.** Runtime → Change runtime type → **T4 GPU**. Everything we need (PyTorch, torchvision, timm) is already installed on Colab.

**Step 1. Data.** [Imagewoof](https://github.com/fastai/imagenette): 10 dog breeds from ImageNet, 9,025 training and 3,929 validation photos, **at full size** (about 1.3 GB, under a minute to download).

In [ ]:
import os, tarfile, time, urllib.request
import torch, timm
from torchvision import datasets

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "Switch to a GPU runtime: Runtime -> Change runtime type -> T4 GPU"

URL = "https://s3.amazonaws.com/fast-ai-imageclas/imagewoof2.tgz"
# we use the full-size image sets on purpose, not the smaller "-320" versions
# can you tell why? (in the deck)
if not os.path.exists("imagewoof2/val"):
    urllib.request.urlretrieve(URL, "imagewoof2.tgz")
    with tarfile.open("imagewoof2.tgz") as t:
        t.extractall(filter="data")
for split in ["train", "val"]:
    print(split, sum(len(f) for _, _, f in os.walk(f"imagewoof2/{split}")), "photos")

**Step 2. Model.** A distilled DeiT-S from the paper (12 blocks, 384 dims, 22M parameters, the DeiT-S⚗ of Table 5), trained on ImageNet at 224×224, together with the exact preprocessing it was trained with.

The model predicts one of 1000 ImageNet classes, so we map each Imagewoof folder (e.g. `n02086240` = Shih-Tzu) to its **ImageNet index**: the position of that breed among the model's 1000 outputs. You will need `to_imagenet` for your fine-tuning too.

<details>
<summary>Running the cell below can ask for your HF_TOKEN secret</summary>
Just press "Grant Access", it's automatically generated by Colab
</details>

In [ ]:
model = timm.create_model("deit_small_distilled_patch16_224", pretrained=True).to(device).eval()
cfg = timm.data.resolve_data_config({}, model=model)  # the preprocessing the model was trained with

# ImageFolder labels the 10 folders 0..9, but the model outputs 1000 ImageNet classes:
# to_imagenet[label] translates a folder label into the ImageNet class index
wnids = list(timm.data.ImageNetInfo().label_names())
breeds = datasets.ImageFolder("imagewoof2/val").classes
to_imagenet = torch.tensor([wnids.index(w) for w in breeds], device=device)
print(cfg)
print("folder -> ImageNet index:", dict(zip(breeds, to_imagenet.tolist())))

**Step 3. The fixed evaluation** (DON'T modify it). `evaluate(model, resolution)` resizes the validation photos for the given resolution exactly as the model's own preprocessing does, and returns the **top-1 accuracy over the 10 breeds**: the model's prediction is the highest-scoring of its 10 dog outputs.

In [ ]:
@torch.no_grad()
def evaluate(model, resolution):
    """The fixed evaluation for task 2. Do not change it; report exactly what it prints.

    model      - any model that takes (batch, 3, resolution, resolution) images and returns 1000 logits
    resolution - the input size: 224 for the original model, e.g. 384 for yours
    returns: accuracy - top-1 over the 10 breeds, on the 3,929 Imagewoof validation photos
    """
    model.eval()
    tf = timm.data.create_transform(**dict(cfg, input_size=(3, resolution, resolution)))
    loader = torch.utils.data.DataLoader(datasets.ImageFolder("imagewoof2/val", transform=tf),
                                         batch_size=64, num_workers=2)
    correct = n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(x)
        correct += (logits[:, to_imagenet].argmax(-1) == y).sum().item()  # best of the 10 breeds
        n += len(y)
    return correct / n

**Step 4. Baseline.** Takes about half a minute. You should get **top-1 ≈ 97.20%** at 224. If not, something in the setup went wrong.

In [ ]:
print(f"baseline DeiT-S distilled @224: top-1 {evaluate(model, 224):.2%}")

# Before you start

Think about **what exactly breaks in the model when you pass it higher-resolution images?** What's the simplest, most effective way to fix that?

Training photos are in `imagewoof2/train` (same folder structure as `val`).

# Your solution

Make the model run at 384, fine-tune within the budget, and collect the evidence for your answer. Good luck!

In [ ]:
# your_model = ...
# print(f"ours @384: top-1 {evaluate(your_model, 384):.2%}")